# Adversarial BERT with FGM & FRAT — Full Dataset Training with Cross-Validation

This notebook trains the proposed FGM-FRAT Adversarial BERT model on each of the five
fake-news datasets using 5-fold cross-validation on the full dataset (no subsets).

## Methodology & Hyperparameters (Qin & Zhang, 2024)
1. **FGM (Fast Gradient Method)**: Embedding-space perturbation with $\epsilon = 0.8$.
2. **FRAT (Feature Regularization)**: $L_2$ regularization between clean and perturbed [CLS] representations with $\lambda = 1.0$.
3. **Re-initialize Top-4 BERT Layers**: Prevents pre-training task specialization.
4. **Freeze Bottom-1 BERT Layer**: Preserves general linguistic features.
5. **Hyperparameters**:
   - `max_length`: 512
   - `batch_size`: 32
   - `learning_rate`: 3e-6
   - `epochs`: 6


# Section 1: Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import random
from torch import nn
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    BertTokenizer, BertModel,
    TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput

from google.colab import drive
drive.mount('/content/drive')

try:
    import torch_xla as torch_xla_pkg
    import torch_xla.core.xla_model as xm
    if not hasattr(torch, "xla"):
        torch.xla = torch_xla_pkg
    _TORCH_XLA_AVAILABLE = True
except Exception:
    xm = None
    _TORCH_XLA_AVAILABLE = False

# Reproducibility
SEED = random.randint(0, 4294967295)
print(f"Random seed: {SEED}")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Section 2: Dataset Paths

In [ ]:
DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/WELFake_processed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/ISOT_processed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv",
}

# Section 3: Dataset Classes

In [ ]:
class FakeNewsDataset(Dataset):
    """Standard dataset for training and evaluation."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

# Section 4: AdversarialBERT Model (with Reinit & Freezing)

In [ ]:
class AdversarialBERT(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1, reinit_layers=4, freeze_layers=1):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)
        
        # 1. Re-initialize the top-K transformer blocks + pooler
        if reinit_layers > 0:
            for layer in self.bert.encoder.layer[-reinit_layers:]:
                for module in layer.modules():
                    if isinstance(module, nn.Linear):
                        module.weight.data.normal_(mean=0.0, std=0.02)
                        if module.bias is not None:
                            module.bias.data.zero_()
                    elif isinstance(module, nn.LayerNorm):
                        module.bias.data.zero_()
                        module.weight.data.fill_(1.0)
            
            if self.bert.pooler is not None:
                for module in self.bert.pooler.modules():
                    if isinstance(module, nn.Linear):
                        module.weight.data.normal_(mean=0.0, std=0.02)
                        if module.bias is not None:
                            module.bias.data.zero_()
            print(f"✓ Re-initialized top {reinit_layers} BERT layers + pooler")
            
        # 2. Freeze the bottom-K transformer blocks + embeddings
        if freeze_layers > 0:
            for param in self.bert.embeddings.parameters():
                param.requires_grad = False
            for layer in self.bert.encoder.layer[:freeze_layers]:
                for param in layer.parameters():
                    param.requires_grad = False
            print(f"✓ Froze embeddings + bottom {freeze_layers} BERT layers")

    def gradient_checkpointing_enable(self, **kwargs):
        self.bert.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, inputs_embeds=None, **kwargs):
        if inputs_embeds is not None:
            out = self.bert(inputs_embeds=inputs_embeds, attention_mask=attention_mask, token_type_ids=token_type_ids)
        else:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_emb = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_emb))
        return logits, cls_emb

# Section 5: FGM-FRAT Custom Trainer

In [ ]:
class FGM_FRAT_Trainer(Trainer):
    """Trainer implementing Fast Gradient Method (FGM) and Feature Regularization (FRAT)."""
    def __init__(self, *args, epsilon=0.8, lambda_frat=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.epsilon = epsilon
        self.lambda_frat = lambda_frat

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Standard classification loss during evaluation, FGM-FRAT during training."""
        inputs = self._prepare_inputs(inputs)
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        token_type_ids = inputs.get("token_type_ids", None)
        labels = inputs["labels"]
        
        # Access underlying BertModel
        raw_model = model.module if hasattr(model, "module") else model
        word_embeddings = raw_model.bert.embeddings.word_embeddings
        
        if not model.training:
            # Standard clean forward pass for validation/evaluation
            logits, _ = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
            return (loss, (loss, logits)) if return_outputs else loss
            
        # --- TRAINING: FGM-FRAT Sequential Backward Pass ---
        # 1. Clean forward pass (leaf tensor for embeddings)
        inputs_embeds = word_embeddings(input_ids).detach().requires_grad_(True)
        logits_clean, cls_clean = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask, token_type_ids=token_type_ids)
        
        loss_fn = nn.CrossEntropyLoss()
        loss_clean = loss_fn(logits_clean, labels)
        
        # Scale clean loss for gradient accumulation before backward
        loss_clean_scaled = loss_clean
        if self.args.gradient_accumulation_steps > 1:
            loss_clean_scaled = loss_clean_scaled / self.args.gradient_accumulation_steps
            
        # First backward pass (computes clean parameter gradients & populates inputs_embeds.grad)
        # This immediately frees the clean pass computational graph from TPU memory.
        if hasattr(self, "accelerator"):
            self.accelerator.backward(loss_clean_scaled)
        else:
            loss_clean_scaled.backward()
            
        # Access the computed gradients on the leaf embedding tensor
        grads = inputs_embeds.grad
        
        # Compute FGM perturbation (detached to prevent second-order gradients)
        norm = torch.norm(grads, dim=-1, keepdim=True)
        delta = (self.epsilon * grads / (norm + 1e-8)).detach()
        
        # Perturbed embeddings (ensuring completely detached from clean graph)
        perturbed_inputs_embeds = inputs_embeds.detach() + delta
        
        # 2. Adversarial forward pass
        logits_adv, cls_adv = model(inputs_embeds=perturbed_inputs_embeds, attention_mask=attention_mask, token_type_ids=token_type_ids)
        
        # Loss terms
        loss_adv = loss_fn(logits_adv, labels)
        frat_loss = torch.mean((cls_adv - cls_clean.detach()) ** 2)
        
        # Combined adversarial and regularization loss
        loss_adv_frat = loss_adv + self.lambda_frat * frat_loss
        
        # Return combined loss to the Trainer
        # Detach loss_clean to prevent secondary backward tracking, but sum it so the Trainer logs the true total loss
        total_loss = loss_clean.detach() + loss_adv_frat
        return (total_loss, (total_loss, logits_clean)) if return_outputs else total_loss

# Section 6: Helper Functions

In [ ]:
def compute_metrics(pred):
    """Compute accuracy, precision, recall, F1."""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary',
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


def load_device():
    if _TORCH_XLA_AVAILABLE and xm is not None:
        try:
            device = xm.xla_device()
            print(f"✓ Using TPU: {device}")
            return device, True
        except Exception:
            pass
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("⚠ Using CPU (Training will be slow!)")
    return device, False


def save_model(model, tokenizer, output_path, use_tpu, device):
    print("\n" + "=" * 60)
    print("MODEL SAVING")
    print("=" * 60)

    os.makedirs(output_path, exist_ok=True)

    if use_tpu:
        model.to("cpu")
        print("Moved model to CPU for saving.")

    torch.save(model.state_dict(), os.path.join(output_path, "adversarial_bert.pt"))
    tokenizer.save_pretrained(output_path)
    print(f"✓ Model saved to {output_path}")

    if use_tpu:
        model.to(device)
        print("Moved model back to TPU.")

    print("\n" + "=" * 60)
    print("ADVERSARIAL BERT FGM-FRAT COMPLETE! 🎉")
    print("=" * 60)

# Section 7: Adversarial Cross-Validation

In [ ]:
def cross_validate_fgm_frat(full_train_texts, full_train_labels,
                              test_dataset, tokenizer, compute_metrics_fn,
                              use_tpu, device, n_splits=5, single_fold=False):
    print("\n" + "=" * 60)
    if single_fold:
        print("STARTING SINGLE-FOLD QUICK EXPERIMENTATION (FGM-FRAT BERT)")
    else:
        print(f"STARTING {n_splits}-FOLD CROSS VALIDATION (FGM-FRAT BERT)")
    print("=" * 60)

    # Pre-tokenize full training set once to save time and RAM
    print("Pre-tokenizing full training pool...")
    full_train_enc = tokenizer(full_train_texts, truncation=True, padding='max_length', max_length=512)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(full_train_labels))

    fold_accuracies = []
    fold_f1_scores = []
    fold_test_results = []
    
    best_f1 = -1.0
    best_model_state = None
    best_fold_idx = -1

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, full_train_labels)):
        print(f"\n--- FOLD {fold + 1} ---")

        train_enc = {k: [v[i] for i in train_idx] for k, v in full_train_enc.items()}
        val_enc = {k: [v[i] for i in val_idx] for k, v in full_train_enc.items()}
        
        train_labels = [full_train_labels[i] for i in train_idx]
        val_labels = [full_train_labels[i] for i in val_idx]

        train_dataset = FakeNewsDataset(train_enc, train_labels)
        val_dataset = FakeNewsDataset(val_enc, val_labels)

        # Fresh model each fold
        model = AdversarialBERT(num_labels=2, dropout=0.1, reinit_layers=4, freeze_layers=1)
        if not use_tpu:
            model.to(device)

        training_kwargs = {
            "output_dir": f'./results_fgm_frat_fold_{fold+1}',
            "num_train_epochs": 3,                # Paper: 6 epochs optimal (set to 3 as requested to make it manageable)
            "per_device_train_batch_size": 32,    # Quadrupled batch size for speed
            "per_device_eval_batch_size": 32,
            "learning_rate": 2e-5,                # Raised to 2e-5 to allow re-initialized layers to learn
            "warmup_ratio": 0.1,                  # 10% warmup steps for stable training
            "gradient_accumulation_steps": 2,     # Effective batch size = 64
            "save_strategy": "steps",
            "save_steps": 256,
            "save_total_limit": 1,
            "bf16": use_tpu,
            "gradient_checkpointing": not use_tpu,
            "report_to": "none",
            "optim": "adamw_torch",
            "logging_dir": './logs',
            "logging_steps": 256,
            "metric_for_best_model": "f1",
            "load_best_model_at_end": True,
            "weight_decay": 0.01,
            "remove_unused_columns": False,
            "label_names": ["labels"],
        }
        if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
            training_kwargs["evaluation_strategy"] = "steps"
            training_kwargs["eval_steps"] = 256
        else:
            training_kwargs["eval_strategy"] = "steps"
            training_kwargs["eval_steps"] = 256

        training_args = TrainingArguments(**training_kwargs)

        trainer = FGM_FRAT_Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics_fn,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
            epsilon=0.8,
            lambda_frat=1.0
        )

        trainer.train()

        # Validation evaluation
        eval_metrics = trainer.evaluate()
        val_f1 = eval_metrics['eval_f1']
        fold_f1_scores.append(val_f1)
        fold_accuracies.append(eval_metrics['eval_accuracy'])
        print(f"Fold {fold+1} Validation - Accuracy: {eval_metrics['eval_accuracy']:.4f}, F1: {val_f1:.4f}")

        # Save best state dict to CPU
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_fold_idx = fold

        # Test set evaluation using PyTorch native inference
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            test_loader = DataLoader(test_dataset, batch_size=32)
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']
                
                logits, _ = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = logits.argmax(-1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(labels.numpy())
                
        precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='binary')
        acc = accuracy_score(all_targets, all_preds)
        test_metrics = {
            'eval_accuracy': acc,
            'eval_f1': f1,
            'eval_precision': precision,
            'eval_recall': recall
        }
        fold_test_results.append(test_metrics)
        print(f"Fold {fold+1} Test     - Accuracy: {test_metrics['eval_accuracy']:.4f}, F1: {test_metrics['eval_f1']:.4f}, Precision: {test_metrics['eval_precision']:.4f}, Recall: {test_metrics['eval_recall']:.4f}")

        # Clean up memory
        del model, trainer, train_dataset, val_dataset
        import gc
        gc.collect()
        if not use_tpu and torch.cuda.is_available():
            torch.cuda.empty_cache()

        if single_fold:
            print("\n⚠ Single fold quick experimentation enabled. Skipping remaining folds...")
            break

    # ---- Summary ----
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    print(f"{'Fold':<6} {'Val Acc':<10} {'Val F1':<10} {'Test Acc':<10} {'Test F1':<10} {'Test Prec':<10} {'Test Rec':<10}")
    print("-" * 66)
    for i in range(len(fold_test_results)):
        t = fold_test_results[i]
        print(f"{i+1:<6} {fold_accuracies[i]:<10.4f} {fold_f1_scores[i]:<10.4f} {t['eval_accuracy']:<10.4f} {t['eval_f1']:<10.4f} {t['eval_precision']:<10.4f} {t['eval_recall']:<10.4f}")

    avg_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    avg_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)

    print(f"\nAverage Validation Accuracy: {avg_acc:.4f} (Std Dev: {std_acc:.4f})")
    print(f"Average Validation F1-score: {avg_f1:.4f} (Std Dev: {std_f1:.4f})")

    best_test = fold_test_results[best_fold_idx]
    print(f"\n★ Best Fold: {best_fold_idx + 1} (Val F1: {fold_f1_scores[best_fold_idx]:.4f})")
    print(f"  Test Results — Accuracy: {best_test['eval_accuracy']:.4f}, F1: {best_test['eval_f1']:.4f}, Precision: {best_test['eval_precision']:.4f}, Recall: {best_test['eval_recall']:.4f}")

    # Re-instantiate the best model on the appropriate device
    best_model = AdversarialBERT(num_labels=2, dropout=0.1, reinit_layers=4, freeze_layers=1)
    best_model.load_state_dict(best_model_state)
    if not use_tpu:
        best_model.to(device)

    return best_model

# Section 8: Cross-Dataset Evaluation

In [ ]:
def cross_dataset_evaluation(model, tokenizer, current_dataset_name, all_datasets_paths, compute_metrics_fn):
    """Evaluate model on all datasets except the one it was trained on.
    Outputs Accuracy, F1, Precision, and Recall for each dataset."""
    device, use_tpu = load_device()

    print("\n" + "!" * 60)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 60)

    class ModelWrapper(nn.Module):
        def __init__(self, inner_model):
            super().__init__()
            self.inner_model = inner_model
        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            outputs = self.inner_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.logits
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels)
            return SequenceClassifierOutput(loss=loss, logits=logits)

    eval_model = ModelWrapper(model)

    results = {}

    for name, path in all_datasets_paths.items():
        if name == current_dataset_name:
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)
        dataset = FakeNewsDataset(encodings, test_labels)

        eval_trainer = Trainer(
            model=eval_model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir="./temp_eval",
                remove_unused_columns=False,
                label_names=["labels"],
                per_device_eval_batch_size=32,
                report_to="none"
            )
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics

        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)

        print(f"  -> {name} Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    return results

# Section 9: Main Training Loop

In [ ]:
def fgm_frat_train_loop(dataset_name, output_path, n_splits=5, single_fold=False):
    dataset_path = DATASETS[dataset_name]

    # Data Loading
    print(f"\nInitialising FGM-FRAT experiment on: {dataset_name}")
    df = pd.read_csv(dataset_path).dropna().reset_index(drop=True)
    print(f"✓ Original dataset loaded: {len(df)} rows")
    print(f"Label distribution:\n{df['label'].value_counts()}")

    all_texts = df['combined_text'].tolist()
    all_labels = df['label'].tolist()

    # 85-15 split
    print("\n" + "=" * 60)
    print("TRAIN / TEST SPLIT (85-15)")
    print("=" * 60)

    indices = list(range(len(df)))
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.15,
        random_state=42,
        stratify=all_labels
    )

    train_texts = [all_texts[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    test_texts = [all_texts[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    print(f"✓ Training pool samples: {len(train_texts)} (85%)")
    print(f"✓ Test set samples:      {len(test_texts)} (15%)")

    # Tokenize test set (standard)
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    print("\nTokenizing held-out test set...")
    test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)
    test_dataset = FakeNewsDataset(test_encodings, test_labels)
    print("✓ Test set tokenized")

    # Cross-Validation
    device, use_tpu = load_device()
    best_model = cross_validate_fgm_frat(
        train_texts, train_labels,
        test_dataset, tokenizer, compute_metrics,
        use_tpu, device, n_splits=n_splits, single_fold=single_fold
    )

    # Save best model
    save_model(best_model, tokenizer, output_path, use_tpu, device)

    # Cross-dataset generalisation
    cross_dataset_evaluation(best_model, tokenizer, dataset_name, DATASETS, compute_metrics)

# Section 9: Quick Experimentation (WELFake & ISOT only, Single Fold)

Run quick single-fold training and downstream generalization evaluation on **WELFake**:

In [ ]:
# WELFake Quick Experiment
fgm_frat_train_loop("WELFake", "/content/drive/MyDrive/models/FGM_FRAT_BERT_WELFake", n_splits=5, single_fold=True)

Run quick single-fold training and downstream generalization evaluation on **ISOT**:

In [ ]:
# ISOT Quick Experiment
fgm_frat_train_loop("ISOT", "/content/drive/MyDrive/models/FGM_FRAT_BERT_ISOT", n_splits=5, single_fold=True)

# Section 10: Full Cross-Validation (All Datasets, 5 Folds)

To run full 5-fold cross-validation on all datasets, uncomment and run the cell below:

In [ ]:
# fgm_frat_train_loop("WELFake", "/content/drive/MyDrive/models/FGM_FRAT_BERT_WELFake", n_splits=5, single_fold=False)
# fgm_frat_train_loop("FakeNewsNet", "/content/drive/MyDrive/models/FGM_FRAT_BERT_FakeNewsNet", n_splits=5, single_fold=False)
# fgm_frat_train_loop("Fake_News_Detection", "/content/drive/MyDrive/models/FGM_FRAT_BERT_Fake_News_Detection", n_splits=5, single_fold=False)
# fgm_frat_train_loop("ISOT", "/content/drive/MyDrive/models/FGM_FRAT_BERT_ISOT", n_splits=5, single_fold=False)
# fgm_frat_train_loop("Fake_News_Classification", "/content/drive/MyDrive/models/FGM_FRAT_BERT_Fake_News_Classification", n_splits=5, single_fold=False)